# 第一章：环境搭建与核心概念

## 学习目标
- 使用 uv 搭建 LlamaIndex 开发环境
- 理解 LlamaIndex 与 LangChain 的设计理念差异
- 掌握核心抽象：Document、LLM、Settings
- 通过 OpenAI 兼容接口接入 Qwen 模型
- 运行第一个 LlamaIndex 程序

## 1. 环境搭建（uv）

本项目使用 **uv** 管理 Python 环境和依赖。LlamaIndex 采用模块化设计，核心包和各集成包分开安装：

```bash
# 安装 LlamaIndex 核心 + OpenAI 兼容 LLM + Embedding
uv add llama-index-core llama-index-llms-openai llama-index-embeddings-openai

# 启动 Jupyter
uv run --with jupyter jupyter lab
```

> **与 LangChain 的对比**：LangChain 也是模块化安装（`langchain-core`、`langchain-openai`），但 LlamaIndex 的拆分更细粒度——每种 LLM、每种 Embedding、每种数据加载器都是独立包。

In [ ]:
# 验证安装 & 版本确认
from importlib.metadata import version

print(f"llama-index-core 版本: {version('llama-index-core')}")
print(f"llama-index-llms-openai 版本: {version('llama-index-llms-openai')}")

: 

## 2. 配置 API Key

本项目通过 OpenAI 兼容接口接入通义千问（Qwen）模型。DashScope 提供了与 OpenAI 格式兼容的 API 端点，因此我们可以直接使用 LlamaIndex 的 OpenAI 集成。

所有密钥存放在项目根目录的 `.env` 文件中（已加入 `.gitignore`，不会提交到版本控制）：

```
Qwen_API_BASE=https://dashscope.aliyuncs.com/compatible-mode/v1
Qwen_API_KEY=sk-...
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("Qwen_API_KEY"):
    print("\u2713 已找到环境变量 Qwen_API_KEY")
else:
    print("\u2717 未找到环境变量 Qwen_API_KEY，请先设置")

## 3. LlamaIndex 是什么？

LlamaIndex 是一个专注于**数据连接与检索**的 LLM 框架。如果说 LangChain 是「LLM 应用的瑞士军刀」，那 LlamaIndex 就是「RAG 领域的专业工具」。

它的核心理念：**让你的私有数据和 LLM 高效连接**。

| | LangChain | LlamaIndex |
|--|--|--|
| **定位** | 通用 LLM 应用框架 | 数据连接与检索专家 |
| **核心能力** | 链式编排、Agent、工具调用 | 索引构建、查询引擎、检索优化 |
| **RAG 支持** | 需手动编排多个组件 | 开箱即用的索引和查询引擎 |
| **接口风格** | Runnable（invoke/stream/batch） | complete/chat + Settings 全局配置 |
| **适合场景** | 复杂 Agent、多步骤工作流 | 文档问答、知识库检索 |
| **关系** | 可以与 LlamaIndex 互补使用 | 可以与 LangChain 互补使用 |

> **一句话总结**：LangChain 擅长编排，LlamaIndex 擅长检索。两者不互斥，很多生产项目会同时使用。

## 4. 第一个 LlamaIndex 程序

先看代码，再解释。用 LlamaIndex 调用 Qwen 模型。

In [ ]:
from llama_index.llms.openai import OpenAI

# 通过 OpenAI 兼容接口接入 Qwen
llm = OpenAI(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

# 第一次调用
response = llm.complete("用一句话解释什么是 RAG")
print(type(response))
print(response.text)

### 刚才发生了什么？

1. **`OpenAI`** 来自 `llama-index-llms-openai`，是 LlamaIndex 对 OpenAI 兼容接口的封装。通过设置 `api_base` 指向 DashScope，就能接入通义千问。
2. **`.complete()`** 是 LlamaIndex 的基础调用方法，接收一个字符串，返回 `CompletionResponse` 对象。
3. **`.text`** 属性获取文本内容——这和 LangChain 中通过 `.content` 获取不同。

> **注意**：LlamaIndex 的 `OpenAI` 类名和 OpenAI 官方 SDK 同名，导入时注意区分来源。

## 5. 对比：LangChain vs LlamaIndex 调用方式

下面把同一件事分别用 LangChain 和 LlamaIndex 实现，直观感受差异。

In [ ]:
# LangChain 的调用方式（你已经熟悉的）
from langchain_openai import ChatOpenAI

llm_lc = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)
response_lc = llm_lc.invoke("用一句话解释什么是 RAG")
print(f"类型: {type(response_lc).__name__}")
print(f"内容: {response_lc.content}")

In [ ]:
# LlamaIndex 的调用方式
response_li = llm.complete("用一句话解释什么是 RAG")
print(f"类型: {type(response_li).__name__}")
print(f"内容: {response_li.text}")

### 调用方式对比

| | LangChain | LlamaIndex |
|--|--|--|
| **类** | `ChatOpenAI` | `OpenAI` |
| **调用方法** | `.invoke()` | `.complete()` / `.chat()` |
| **返回类型** | `AIMessage` | `CompletionResponse` / `ChatResponse` |
| **获取文本** | `.content` | `.text` / `.message.content` |

两者都能完成同样的任务，但接口风格不同。LangChain 追求统一的 Runnable 接口，LlamaIndex 则更贴近原始的 complete/chat 语义。

## 6. Chat 模式与消息类型

除了 `complete()`，LlamaIndex 也支持多轮对话的 `chat()` 模式。

In [ ]:
from llama_index.core.llms import ChatMessage, MessageRole

# 构造多轮对话
messages = [
    ChatMessage(role=MessageRole.SYSTEM, content="你是一位 Python 编程老师，回答要简洁。"),
    ChatMessage(role=MessageRole.USER, content="列表和元组有什么区别？"),
]

response = llm.chat(messages)
print(response.message.content)

### 刚才发生了什么？

- **`ChatMessage`** 是 LlamaIndex 的消息对象，通过 `role` 参数指定角色（SYSTEM / USER / ASSISTANT）。
- **`MessageRole`** 是枚举类型，定义了可用的角色。
- **`.chat()`** 接收消息列表，返回 `ChatResponse`，通过 `.message.content` 获取回复内容。

与 LangChain 的消息类型对比：

| LangChain | LlamaIndex |
|--|--|
| `SystemMessage(content=...)` | `ChatMessage(role=MessageRole.SYSTEM, content=...)` |
| `HumanMessage(content=...)` | `ChatMessage(role=MessageRole.USER, content=...)` |
| `AIMessage`（返回） | `ChatResponse.message`（返回） |

> LangChain 为每种角色定义了独立的类，LlamaIndex 则用统一的 `ChatMessage` + `role` 参数。各有优劣——LangChain 更类型安全，LlamaIndex 更简洁。

## 7. Document 对象

`Document` 是 LlamaIndex 中最基础的数据单元，代表一段文本及其元数据。后续的索引构建、检索查询都以 Document 为起点。

In [ ]:
from llama_index.core import Document

# 创建文档
doc = Document(
    text="LlamaIndex 是一个专注于数据连接和检索的 LLM 框架。",
    metadata={"source": "tutorial", "chapter": 1},
)

print(f"文本: {doc.text}")
print(f"元数据: {doc.metadata}")
print(f"文档ID: {doc.doc_id}")

### Document 对比

- LlamaIndex 的 `Document` 使用 `text` 字段存储文本（LangChain 用 `page_content`）。
- 两者都有 `metadata` 字典存储元数据。
- LlamaIndex 会自动生成 `doc_id` 作为唯一标识，方便后续索引管理。

| | LangChain | LlamaIndex |
|--|--|--|
| **文本字段** | `page_content` | `text` |
| **元数据** | `metadata` | `metadata` |
| **唯一标识** | 无 | `doc_id`（自动生成） |

## 8. Settings 全局配置

LlamaIndex 提供了一个 `Settings` 对象来管理全局配置。这是它与 LangChain 最大的风格差异之一。

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# 配置全局 LLM
Settings.llm = llm

# 配置全局 Embedding 模型
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

print(f"全局 LLM: {Settings.llm.model}")
print(f"全局 Embedding: {Settings.embed_model.model_name}")

### 刚才发生了什么？

`Settings` 是 LlamaIndex 的**全局配置中心**。设置一次后，后续创建的所有索引、查询引擎都会自动使用这里配置的 LLM 和 Embedding 模型。

这是 LlamaIndex 与 LangChain 的一个关键设计差异：

- **LangChain**：每个组件需要显式传入 LLM，例如 `SomeChain(llm=llm)`。好处是清晰可控，坏处是代码重复。
- **LlamaIndex**：通过 `Settings` 全局配置，所有组件自动获取。好处是少写代码，坏处是隐式依赖。

当然，LlamaIndex 也支持在具体组件中覆盖全局配置，所以这不是「全有或全无」的选择。

> **类比**：`Settings` 就像 Python 的 `logging.basicConfig()`——配一次，全局生效，需要时再单独覆盖。

## 9. 流式输出

和 LangChain 一样，LlamaIndex 也支持流式输出，适合长文本生成时逐步显示结果。

In [ ]:
# 流式调用
for chunk in llm.stream_complete("用三句话介绍 LlamaIndex 的优势"):
    print(chunk.delta, end="", flush=True)
print()

### 流式输出说明

- **`.stream_complete()`** 返回一个生成器，逐步产出 `CompletionResponse` 对象。
- **`.delta`** 属性是每次新增的文本片段（增量内容）。
- 使用 `end=""` 和 `flush=True` 让输出逐字显示，不换行。

与 LangChain 对比：
- LangChain 用 `.stream()` 方法，返回 `AIMessageChunk`，通过 `.content` 获取增量。
- LlamaIndex 用 `.stream_complete()`，返回 `CompletionResponse`，通过 `.delta` 获取增量。

## 10. 核心概念总览

| 概念 | 说明 |
|------|------|
| **OpenAI (LLM)** | LLM 封装，支持 OpenAI 兼容接口 |
| **Document** | 文档对象，包含 text、metadata、doc_id |
| **Settings** | 全局配置，设定默认 LLM 和 Embedding 模型 |
| **complete()** | 文本补全调用 |
| **chat()** | 多轮对话调用 |
| **stream_complete()** | 流式文本补全 |

LlamaIndex 编程模型一句话概括：

> **配置 Settings → 加载 Document → 构建 Index → 创建 QueryEngine → 查询**

本章完成了前两步，后续章节逐步展开。

## 总结

本章学习了：
- 使用 uv 搭建 LlamaIndex 开发环境
- LlamaIndex vs LangChain 的定位差异：数据检索专家 vs 通用 LLM 框架
- 通过 OpenAI 兼容接口接入 Qwen 模型
- complete() 和 chat() 两种调用方式
- Document 对象和 Settings 全局配置
- 流式输出 stream_complete()

## 下一章预告

**第二章：文档加载与索引构建** —— 学习如何从文件加载文档、切分为节点，并构建 VectorStoreIndex。一行代码就能搭建一个可用的 RAG 系统。